# JPX Tokyo Stock Exchange Prediction — Group 7

**Objective:** Predict 2-day forward returns for ~2000 Japanese stocks and rank them daily.  
**Models:** XGBoost (prices + financials), XGBoost (+ options), Logistic Regression, Random Forest  
**Metric:** Annualised Sharpe Ratio on long-top-200 / short-bottom-200 spread

## 0. Setup & Imports

In [ ]:
!pip install kaggle xgboost scikit-learn plotly scipy --quiet

import os, json, time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import xgboost as xgb
from scipy.stats import mstats
import scipy.stats as stats
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error

pd.set_option('display.max_columns', None)
print('✅ All imports done')

## 1. Download Data

In [ ]:
# Kaggle credentials
kaggle_username = 'victorszee'
kaggle_key      = '1a4058f62eee1f923e1ee10aa917de49'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
creds_path = os.path.expanduser('~/.kaggle/kaggle.json')
with open(creds_path, 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(creds_path, 0o600)
print('✅ Credentials saved')

In [ ]:
!kaggle competitions download -c jpx-tokyo-stock-exchange-prediction

import zipfile
with zipfile.ZipFile('jpx-tokyo-stock-exchange-prediction.zip', 'r') as z:
    z.extractall('jpx_data')
    print('✅ Extracted:', len(z.namelist()), 'files')

## 2. Load & Clean Data

In [ ]:
# ── Stock prices
price_cols = ['Date', 'SecuritiesCode', 'Open', 'High', 'Low', 'Close', 'Volume', 'Target']
df_prices = pd.read_csv('jpx_data/train_files/stock_prices.csv',
                         usecols=price_cols, dtype={'SecuritiesCode': 'int32'}, low_memory=False)
df_prices['Date'] = pd.to_datetime(df_prices['Date'])
df_prices = df_prices.sort_values(['SecuritiesCode', 'Date'])
for col in ['Open', 'High', 'Low', 'Close', 'Volume', 'Target']:
    df_prices[col] = pd.to_numeric(df_prices[col], errors='coerce').astype('float32')
df_prices = df_prices.dropna(subset=['Close', 'Open', 'High', 'Low'])
print('✅ Prices:', df_prices.shape)

# ── Financials
fin_cols = ['Date', 'SecuritiesCode', 'NetSales', 'OperatingProfit',
            'EarningsPerShare', 'EquityToAssetRatio', 'TotalAssets', 'Equity']
df_fin = pd.read_csv('jpx_data/train_files/financials.csv',
                      usecols=fin_cols, on_bad_lines='skip', low_memory=False)
df_fin['Date'] = pd.to_datetime(df_fin['Date'])
df_fin = df_fin.dropna(subset=['SecuritiesCode'])
df_fin['SecuritiesCode'] = df_fin['SecuritiesCode'].astype('int32')
df_fin = df_fin.sort_values(['SecuritiesCode', 'Date'])
df_fin = df_fin.set_index('SecuritiesCode')
df_fin = df_fin.groupby('SecuritiesCode', group_keys=False).apply(lambda x: x.ffill()).fillna(0)
df_fin = df_fin.reset_index()
df_fin['Date'] = pd.to_datetime(df_fin['Date'])
df_fin['SecuritiesCode'] = df_fin['SecuritiesCode'].astype('int32')
print('✅ Financials:', df_fin.shape)

# ── Stock list
df_stock_list = pd.read_csv('jpx_data/stock_list.csv', low_memory=False)
print('✅ Stock list:', df_stock_list.shape)

In [ ]:
# Missing value check
print('=== Prices missing values ===')
print(df_prices.isnull().sum().sort_values(ascending=False).head(10))

## 3. Exploratory Data Analysis

In [ ]:
# Target distribution
target_data = df_prices['Target'].dropna().clip(-0.1, 0.1)
fig = go.Figure(go.Histogram(x=target_data, nbinsx=100, marker_color='steelblue'))
fig.update_layout(title='Target Distribution (clipped ±10%)',
                  xaxis_title='2-day forward return', yaxis_title='Count', height=400)
fig.show()

## 4. Feature Engineering

In [ ]:
# Date features
df_prices['Year']    = df_prices['Date'].dt.year
df_prices['Month']   = df_prices['Date'].dt.month
df_prices['Day']     = df_prices['Date'].dt.day
df_prices['Weekday'] = df_prices['Date'].dt.weekday

# Price momentum
df_prices['Return_1d'] = df_prices.groupby('SecuritiesCode')['Close'].pct_change(1).astype('float32')
df_prices['Return_5d'] = df_prices.groupby('SecuritiesCode')['Close'].pct_change(5).astype('float32')
df_prices['MA_5']      = df_prices.groupby('SecuritiesCode')['Close'].transform(lambda x: x.rolling(5).mean()).astype('float32')
df_prices['MA_20']     = df_prices.groupby('SecuritiesCode')['Close'].transform(lambda x: x.rolling(20).mean()).astype('float32')
df_prices['MA_ratio']  = (df_prices['Close'] / df_prices['MA_5']).astype('float32')

# Volume
df_prices['Vol_MA5']   = df_prices.groupby('SecuritiesCode')['Volume'].transform(lambda x: x.rolling(5).mean()).astype('float32')
df_prices['Vol_ratio'] = (df_prices['Volume'] / df_prices['Vol_MA5']).astype('float32')

print('✅ Features done:', df_prices.shape)
print(f'RAM: {df_prices.memory_usage(deep=True).sum()/1e6:.1f} MB')

## 5. Model 1 — XGBoost: Prices + Financials

XGBoost with 500 decision trees trained on 4 years of historical data (2017–2020) across ~2000 Japanese stocks. Uses 8 features combining technical signals (momentum, volume) and fundamentals (operating profit, EPS). Predicts 2-day forward return, then ranks all stocks daily.

In [ ]:
# Merge prices + financials
df_merged = pd.merge(df_prices, df_fin, on=['SecuritiesCode', 'Date'], how='left')

fin_cols = ['NetSales', 'OperatingProfit', 'EarningsPerShare', 'EquityToAssetRatio']
for col in fin_cols:
    if col in df_merged.columns:
        df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce')

df_merged = df_merged.sort_values(['SecuritiesCode', 'Date'])
for col in fin_cols:
    df_merged[col] = df_merged.groupby('SecuritiesCode')[col].transform(
        lambda x: x.ffill().fillna(0)
    )
df_merged = df_merged.fillna(0)
print('✅ Merged:', df_merged.shape)

In [ ]:
# Features + train/test split
feature_cols = ['Return_1d', 'Return_5d', 'MA_ratio', 'Vol_ratio',
                'NetSales', 'OperatingProfit', 'EarningsPerShare', 'EquityToAssetRatio']
feature_cols = [c for c in feature_cols if c in df_merged.columns]
print('Features:', feature_cols)

df_model = df_merged.dropna(subset=['Target'])
train = df_model[df_model['Date'] < '2021-01-01']
test  = df_model[df_model['Date'] >= '2021-01-01']

X_train, y_train = train[feature_cols], train['Target']
X_test,  y_test  = test[feature_cols],  test['Target']
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')

In [ ]:
# Train + carbon tracking
AVERAGE_POWER_KW = 0.25
CARBON_INTENSITY = 0.2

start = time.time()
model1 = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                           verbosity=0, tree_method='hist')
model1.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
end = time.time()

runtime_h = (end - start) / 3600
kwh  = runtime_h * AVERAGE_POWER_KW
co2e = kwh * CARBON_INTENSITY
print(f'✅ Model 1 trained | Runtime: {end-start:.1f}s | CO₂: {co2e:.6f} kg')

# Predict + rank
test = test.copy()
test['PredictedReturn'] = model1.predict(X_test)
test['Rank'] = test.groupby('Date')['PredictedReturn'].rank(ascending=False).astype(int) - 1
print('✅ Rankings done')
print(test[['Date', 'SecuritiesCode', 'PredictedReturn', 'Rank']].head(10))

In [ ]:
# Evaluation metrics
y_pred = test['PredictedReturn']
y_true = test['Target']

rmse    = mean_squared_error(y_true, y_pred) ** 0.5
mae     = mean_absolute_error(y_true, y_pred)
dir_acc = ((test['PredictedReturn'] > 0) == (test['Target'] > 0)).mean() * 100
spearman, pvalue = stats.spearmanr(y_pred, y_true)

# Sharpe ratio
test_sorted   = test.sort_values(['Date', 'Rank'])
daily_returns = []
for date, group in test_sorted.groupby('Date'):
    top200 = group[group['Rank'] <= 199]['Target'].mean()
    bot200 = group[group['Rank'] >= (group['Rank'].max() - 199)]['Target'].mean()
    daily_returns.append(top200 - bot200)
daily_returns = pd.Series(daily_returns)
sharpe = daily_returns.mean() / daily_returns.std() * (252 ** 0.5)

corr = test[['PredictedReturn', 'Target']].corr()
print('Correlation between predicted and actual return:')
print(corr)

print('\n====== MODEL 1 EVALUATION SUMMARY ======')
print(f'RMSE:                  {rmse:.6f}')
print(f'MAE:                   {mae:.6f}')
print(f'Directional Accuracy:  {dir_acc:.2f}%  (50%=random, 55%+=decent)')
print(f'Spearman Correlation:  {spearman:.4f}  (p={pvalue:.4f})')
print(f'Sharpe Ratio:          {sharpe:.4f}')

In [ ]:
# Plot 1 — Feature importance
importance = model1.get_booster().get_fscore()
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:8]
feats  = [x[0] for x in importance_sorted]
scores = [x[1] for x in importance_sorted]

fig = go.Figure(go.Bar(x=scores, y=feats, orientation='h', marker_color='steelblue'))
fig.update_layout(title='Model 1 — Feature Importance', xaxis_title='F Score',
                  yaxis_title='Feature', height=400)
fig.show()

# Plot 2 — Cumulative return
cumulative = (1 + daily_returns).cumprod()
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=cumulative.values, mode='lines',
                           name='Portfolio', line=dict(color='steelblue')))
fig2.add_hline(y=1, line_dash='dash', line_color='red', annotation_text='Baseline')
fig2.update_layout(title='Model 1 — Cumulative Portfolio Return',
                   xaxis_title='Trading Days', yaxis_title='Cumulative Return', height=400)
fig2.show()

## 6. Model 2 — XGBoost: Prices + Financials + Options

Extends Model 1 by adding options market data as additional features.

In [ ]:
# Load & clean options
df_options = pd.read_csv('jpx_data/train_files/options.csv', low_memory=False)
df_options['Date'] = pd.to_datetime(df_options['Date'], errors='coerce')

# Parse contract month
df_options = df_options.dropna(subset=['ContractMonth']).copy()
df_options['contract_clean'] = pd.to_numeric(df_options['ContractMonth'], errors='coerce')
df_options = df_options.dropna(subset=['contract_clean'])
df_options['contract_clean'] = df_options['contract_clean'].astype(int)
df_options['contract_date']  = pd.to_datetime(
    df_options['contract_clean'].astype(str), format='%Y%m', errors='coerce'
)

# Convert numeric cols
for col in ['DaySessionHigh', 'DaySessionLow', 'NightSessionHigh', 'NightSessionLow',
            'DaySessionClose', 'NightSessionClose', 'TradingVolume', 'TradingValue']:
    if col in df_options.columns:
        df_options[col] = pd.to_numeric(df_options[col], errors='coerce')

# Filter invalid rows
df_options = df_options[df_options['DaySessionHigh'] >= df_options['DaySessionLow']]

# Feature engineering
df_options['day_range']   = df_options['DaySessionHigh']  - df_options['DaySessionLow']
df_options['night_range'] = df_options['NightSessionHigh'] - df_options['NightSessionLow']
df_options['was_traded']  = (
    ((df_options['DaySessionClose'] > 0) | (df_options['NightSessionClose'] > 0)) &
    (df_options['TradingVolume'] > 0)
)
df_options['large_trade'] = df_options['TradingValue'] > df_options['TradingValue'].quantile(0.99)

# Winsorize
df_options['TradingValue_winsorized'] = mstats.winsorize(
    df_options['TradingValue'].fillna(0), limits=[0, 0.001]
)

# Drop uninformative cols
drop_cols = [c for c in ['Dividend', 'DividendRate', 'TradingValue', 'log_trading_value']
             if c in df_options.columns]
df_options.drop(columns=drop_cols, inplace=True)

print('✅ Options loaded:', df_options.shape)

In [ ]:
# Merge options into df_merged
if 'OptionsCode' in df_options.columns:
    df_options = df_options.rename(columns={'OptionsCode': 'SecuritiesCode'})
df_options['Date'] = pd.to_datetime(df_options['Date'])

# Keep highest-volume option per stock per day
df_options_rep = df_options.loc[
    df_options.groupby(['SecuritiesCode', 'Date'])['TradingVolume'].idxmax()
].reset_index(drop=True)

df_merged2 = pd.merge(df_merged, df_options_rep, on=['SecuritiesCode', 'Date'], how='left')
df_merged2 = df_merged2.fillna(0)
print('✅ Merged with options:', df_merged2.shape)

In [ ]:
# Train Model 2
feature_cols2 = ['Return_1d', 'Return_5d', 'MA_ratio', 'Vol_ratio',
                 'NetSales', 'OperatingProfit', 'EarningsPerShare', 'EquityToAssetRatio',
                 'day_range', 'night_range', 'was_traded', 'large_trade']
feature_cols2 = [c for c in feature_cols2 if c in df_merged2.columns]
print('Features:', feature_cols2)

df_model2 = df_merged2.dropna(subset=['Target'])
train2 = df_model2[df_model2['Date'] < '2021-01-01']
test2  = df_model2[df_model2['Date'] >= '2021-01-01']

X_train2, y_train2 = train2[feature_cols2], train2['Target']
X_test2,  y_test2  = test2[feature_cols2],  test2['Target']
print(f'Train: {len(X_train2)} | Test: {len(X_test2)}')

start = time.time()
model2 = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                           verbosity=0, tree_method='hist')
model2.fit(X_train2, y_train2, eval_set=[(X_test2, y_test2)], verbose=False)
end = time.time()

kwh  = ((end-start)/3600) * AVERAGE_POWER_KW
co2e = kwh * CARBON_INTENSITY
print(f'✅ Model 2 trained | Runtime: {end-start:.1f}s | CO₂: {co2e:.6f} kg')

test2 = test2.copy()
test2['PredictedReturn'] = model2.predict(X_test2)
test2['Rank'] = test2.groupby('Date')['PredictedReturn'].rank(ascending=False).astype(int) - 1

In [ ]:
# Model 2 evaluation
rmse2    = mean_squared_error(test2['Target'], test2['PredictedReturn']) ** 0.5
mae2     = mean_absolute_error(test2['Target'], test2['PredictedReturn'])
dir_acc2 = ((test2['PredictedReturn'] > 0) == (test2['Target'] > 0)).mean() * 100
spearman2, pvalue2 = stats.spearmanr(test2['PredictedReturn'], test2['Target'])

daily_returns2 = []
for date, group in test2.sort_values(['Date','Rank']).groupby('Date'):
    top200 = group[group['Rank'] <= 199]['Target'].mean()
    bot200 = group[group['Rank'] >= (group['Rank'].max()-199)]['Target'].mean()
    daily_returns2.append(top200 - bot200)
daily_returns2 = pd.Series(daily_returns2)
sharpe2 = daily_returns2.mean() / daily_returns2.std() * (252**0.5)

print('====== MODEL 2 EVALUATION SUMMARY ======')
print(f'RMSE:                  {rmse2:.6f}')
print(f'MAE:                   {mae2:.6f}')
print(f'Directional Accuracy:  {dir_acc2:.2f}%')
print(f'Spearman Correlation:  {spearman2:.4f}  (p={pvalue2:.4f})')
print(f'Sharpe Ratio:          {sharpe2:.4f}')

In [ ]:
# Model 2 plots
importance2 = model2.get_booster().get_fscore()
imp2_sorted = sorted(importance2.items(), key=lambda x: x[1], reverse=True)[:10]

fig = go.Figure(go.Bar(x=[x[1] for x in imp2_sorted], y=[x[0] for x in imp2_sorted],
                        orientation='h', marker_color='steelblue'))
fig.update_layout(title='Model 2 — Feature Importance', xaxis_title='F Score', height=400)
fig.show()

cumulative2 = (1 + daily_returns2).cumprod()
fig2 = go.Figure()
fig2.add_trace(go.Scatter(y=cumulative2.values, mode='lines', name='Portfolio',
                           line=dict(color='steelblue')))
fig2.add_hline(y=1, line_dash='dash', line_color='red', annotation_text='Baseline')
fig2.update_layout(title='Model 2 — Cumulative Portfolio Return',
                   xaxis_title='Trading Days', yaxis_title='Cumulative Return', height=400)
fig2.show()

## 7. Model 3 — Logistic Regression (Direction Classification)

Classifies whether each stock will go up or down (binary), using price-based features.

In [ ]:
# Prepare data for classification models
dat = df_prices.copy()
dat = dat.dropna()
dat['Direction']    = (dat['Target'] > 0).astype(int)
dat['Return']       = dat.groupby('SecuritiesCode')['Close'].pct_change()
dat['PriceRange']   = (dat['High'] - dat['Low']) / dat['Open']
dat['MA5']          = dat.groupby('SecuritiesCode')['Close'].transform(lambda x: x.rolling(5).mean())
dat['VolumeChange'] = dat.groupby('SecuritiesCode')['Volume'].pct_change()
dat = dat.dropna()

features = ['Open', 'High', 'Low', 'Close', 'Volume', 'PriceRange', 'Return', 'MA5', 'VolumeChange']
X = dat[features]
y = dat['Direction']

X_train3, X_test3, y_train3, y_test3 = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train3)} | Test: {len(X_test3)}')

In [ ]:
# Logistic Regression
start = time.time()
model3 = LogisticRegression(max_iter=1000)
model3.fit(X_train3, y_train3)
end = time.time()

kwh  = ((end-start)/3600) * AVERAGE_POWER_KW
co2e = kwh * CARBON_INTENSITY
print(f'Runtime: {end-start:.2f}s | CO₂: {co2e:.6f} kg')

y_pred3 = model3.predict(X_test3)
rmse3   = root_mean_squared_error(y_test3, y_pred3)
up_pct  = len(y_pred3[y_pred3 == 1]) / len(y_pred3) * 100
print(f'RMSE: {rmse3:.4f}')
print(f'Predicted UP: {up_pct:.1f}%')

In [ ]:
# Logistic Regression predicted probabilities plot
y_prob3 = model3.predict_proba(X_test3)[:,1]

fig = go.Figure(go.Scatter(
    x=list(range(len(y_test3))),
    y=y_prob3,
    mode='markers',
    marker=dict(color=y_test3.values, colorscale='RdBu', opacity=0.5, size=3)
))
fig.update_layout(title='Logistic Regression — Predicted Probabilities',
                  xaxis_title='Sample Index', yaxis_title='P(Up)', height=400)
fig.show()

## 8. Model 4 — Random Forest Classifier

In [ ]:
# Random Forest
start = time.time()
model4 = RandomForestClassifier(n_estimators=50, max_depth=8, max_features='log2',
                                 random_state=42, n_jobs=-1)
model4.fit(X_train3, y_train3)
end = time.time()

y_pred4 = model4.predict(X_test3)
kwh  = ((end-start)/3600) * AVERAGE_POWER_KW
co2e = kwh * CARBON_INTENSITY
rmse4 = root_mean_squared_error(y_test3, y_pred4)

print(f'Runtime: {end-start:.2f}s | CO₂: {co2e:.6f} kg')
print(f'RMSE: {rmse4:.4f}')

In [ ]:
# Random Forest predicted probabilities plot
y_prob4 = model4.predict_proba(X_test3)[:,1]

fig = go.Figure(go.Scatter(
    x=list(range(len(y_test3))),
    y=y_prob4,
    mode='markers',
    marker=dict(color=y_test3.values, colorscale='RdBu', opacity=0.5, size=3)
))
fig.update_layout(title='Random Forest — Predicted Probabilities',
                  xaxis_title='Sample Index', yaxis_title='P(Up)', height=400)
fig.show()

## 9. Model Comparison Summary

In [ ]:
print('=' * 55)
print(f'{"Model":<35} {"Sharpe":>8} {"Dir Acc":>10}')
print('=' * 55)
print(f'{"Model 1 — XGBoost (prices + fin)":<35} {sharpe:>8.4f} {dir_acc:>9.2f}%')
print(f'{"Model 2 — XGBoost (+ options)":<35} {sharpe2:>8.4f} {dir_acc2:>9.2f}%')
print(f'{"Model 3 — Logistic Regression":<35} {"N/A":>8} {"N/A":>10}')
print(f'{"Model 4 — Random Forest":<35} {"N/A":>8} {"N/A":>10}')
print('=' * 55)

## 10. Environmental Impact

In [ ]:
# Total CO2 estimate across all models
# (individual values printed during each model training above)
print('Environmental Impact Summary')
print('Assumed: 0.25 kW CPU, 0.2 kg CO₂/kWh grid intensity')
print('Carbon emissions were tracked per model during training.')